<a href="https://colab.research.google.com/github/mc-castro/clinicaltrials-ia-thesis/blob/mc-castro%2Fdevelop/notebooks/parsing_protocolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [ ]:
import requests
import json
import re
import pandas as pd
from typing import List, Dict, Any, Tuple
import time
import os
import ast
from tqdm import tqdm
import typing_extensions as typing
import google.generativeai as genai
from google.colab import userdata, drive


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Configs
genai.configure(api_key=userdata.get('ParsingGeminiAPI'))
MODEL_NAME = 'gemini-2.5-flash'
model = genai.GenerativeModel(MODEL_NAME)

## Parsing Studies

In [ ]:
# @title API Clinical Trails
def get_clinical_studies_data(diseases: List[str], status: str = 'COMPLETED', page_size: int = 10) -> pd.DataFrame:
    """
    Acquires clinical study data from the ClinicalTrials.gov API based on a list of diseases and a study status.

    Args:
        diseases (List[str]): List of disease terms for the search query.
        status (str): Overall study status to filter (e.g., 'COMPLETED').
        page_size (int): Maximum number of studies to return per request (max. 1000).

    Returns:
        pd.DataFrame: A DataFrame where each row is a study and columns are the elements extracted from the API (NCT ID, Title, Criteria).
                      Returns an empty DataFrame if no data is found or an error occurs.
    """

    query_terms = ' OR '.join([f'"{d}"' for d in diseases])

    params = {
        'query.term': query_terms,
        'filter.overallStatus': status,
        'countTotal': 'true',
        'pageSize': page_size
    }

    try:
        response = requests.get(API_URL, params=params)
        response.raise_for_status()

        data: Dict[str, Any] = response.json()

        studies_data: List[Dict[str, str]] = []

        if 'studies' in data:
            studies: List[Dict[str, Any]] = data['studies']

            for study in studies:
                study_info = study.get('protocolSection', {})

                # Modules of interest
                identification_module = study_info.get('identificationModule', {})
                eligibility_module = study_info.get('eligibilityModule', {})

                # Extract fields
                nct_id = identification_module.get('nctId', '')
                official_title = identification_module.get('officialTitle', '')
                eligibility_criteria = eligibility_module.get('eligibilityCriteria', '')

                # Append study data to the list
                studies_data.append({
                    'nct_id': nct_id,
                    'official_title': official_title,
                    'eligibility_criteria': eligibility_criteria
                })

            # Create the pandas DataFrame and set 'nct_id' as the index
            df = pd.DataFrame(studies_data)
            df.set_index('nct_id', inplace=True)

            return df

        else:
            print("The API response does not contain the 'studies' key. Returning empty DataFrame.")
            return pd.DataFrame()

    except requests.exceptions.HTTPError as errh:
        print(f"Request Error (HTTP): {errh}")
        return pd.DataFrame()
    except requests.exceptions.ConnectionError as errc:
        print(f"Request Error (Connection): {errc}")
        return pd.DataFrame()
    except requests.exceptions.Timeout as errt:
        print(f"Request Error (Timeout): {errt}")
        return pd.DataFrame()
    except requests.exceptions.RequestException as e:
        print(f"An unknown request error occurred: {e}")
        return pd.DataFrame()

In [ ]:
# @title Extract with Regex
def process_criteria_robust(criteria_text: str) -> Tuple[List[str], List[str]]:
    """
    Splits clinical trial criteria into inclusion and exclusion lists.
    Fixes the 'last item only' bug and handles multiple list markers (1., *, -, etc).
    """
    # 1. Handle null or non-string inputs
    if not isinstance(criteria_text, str) or not criteria_text.strip():
        return [], []

    # Normalize line breaks and remove extra whitespace
    text = criteria_text.replace('\r', '').replace('\n', ' ').strip()

    # 2. Extract sections using non-greedy regex
    # Capture everything between "Inclusion Criteria:" and "Exclusion Criteria:" (or end of string)
    inclusion_match = re.search(r'Inclusion Criteria:?\s*(.*?)(?=Exclusion Criteria:?|$)', text, re.IGNORECASE | re.DOTALL)
    exclusion_match = re.search(r'Exclusion Criteria:?\s*(.*)', text, re.IGNORECASE | re.DOTALL)

    inclusion_raw = inclusion_match.group(1).strip() if inclusion_match else ""
    exclusion_raw = exclusion_match.group(1).strip() if exclusion_match else ""

    # 3. Helper function to split text into list items
    def split_into_items(raw_text: str) -> List[str]:
        if not raw_text:
            return []

        # This regex looks for:
        # - Asterisks (*) or Dashes (-)
        # - Numbers followed by dot, dash or paren (1., 1-, 1))
        # - Letters followed by paren (a))
        # - Double spaces or specific "Additional" headers
        marker_pattern = r'\s*[\*\-]\s+|\s*\d+[\.\-\)]\s+|\s*[a-z]\)\s+|(?=Additional Inclusion Criteria)'

        # Split by the markers
        items = re.split(marker_pattern, raw_text)

        # Clean each item: remove extra dots at the end, leading/trailing spaces
        cleaned_items = []
        for item in items:
            clean = item.strip()
            # Ignore very short strings or headers
            if len(clean) > 3 and not clean.lower().endswith('criteria:'):
                # Remove common prefixes like "and ", "or "
                clean = re.sub(r'^(and/or|and|or)\s+', '', clean, flags=re.IGNORECASE)
                cleaned_items.append(clean)

        return cleaned_items

    inclusion_list = split_into_items(inclusion_raw)
    exclusion_list = split_into_items(exclusion_raw)

    return inclusion_list, exclusion_list

In [ ]:
# @title Extract with Gemini

# # Define the structured output format
# class ClinicalCriteria(typing.TypedDict):
#     inclusion_criteria: list[str]
#     exclusion_criteria: list[str]

# # --- 2. EXTRACTION FUNCTION ---
# def extract_criteria_gemini(text: str) -> dict:
#     """
#     Sends raw eligibility text to Gemini and returns a structured dictionary
#     containing two lists: inclusion and exclusion criteria.
#     """
#     if not text or pd.isna(text):
#         return {"inclusion_criteria": [], "exclusion_criteria": []}

#     prompt = f"""
#     You are a medical data specialist. Analyze the 'Eligibility Criteria' section below.
#     Tasks:
#     1. Distinguish between 'Inclusion' and 'Exclusion' sections.
#     2. Split each section into a clean list of individual criteria.
#     3. Remove all list markers (e.g., '1.', '*', '-', 'a)').
#     4. Ignore meta-text like 'see below' or '(refer to exclusion criteria)'.

#     RAW TEXT:
#     {text}
#     """

#     try:
#         response = model.generate_content(
#             prompt,
#             generation_config=genai.GenerationConfig(
#                 response_mime_type="application/json",
#                 response_schema=ClinicalCriteria
#             )
#         )
#         return json.loads(response.text)
#     except Exception as e:
#         print(f"API Error: {e}")
#         return {"inclusion_criteria": [], "exclusion_criteria": []}

# # --- 3. BATCH PROCESSING WITH CHECKPOINT ---
# def process_clinical_dataset(df: pd.DataFrame, output_filename="processed_studies.csv"):
#     """
#     Processes the entire dataframe row by row with a progress bar.
#     Saves a checkpoint every 5 rows to prevent data loss.
#     """
#     # Check if a checkpoint file already exists to resume progress
#     if os.path.exists(output_filename):
#         df_processed = pd.read_csv(output_filename)
#         processed_count = len(df_processed)
#         print(f"Resuming from index {processed_count}...")
#     else:
#         df_processed = pd.DataFrame()
#         processed_count = 0

#     # Main loop for the remaining rows
#     for i in tqdm(range(processed_count, len(df))):
#         row = df.iloc[i].copy()

#         # Call the AI extraction
#         extraction_result = extract_criteria_gemini(row['eligibility_criteria'])

#         # Store lists as JSON strings (CSV requirement)
#         row['inclusion_criteria'] = json.dumps(extraction_result['inclusion_criteria'])
#         row['exclusion_criteria'] = json.dumps(extraction_result['exclusion_criteria'])

#         # Append to the processed dataframe
#         df_processed = pd.concat([df_processed, pd.DataFrame([row])], ignore_index=True)

#         # Save checkpoint every 5 rows
#         if (i + 1) % 5 == 0:
#             df_processed.to_csv(output_filename, index=False)

#         # Rate Limiting: 15 requests per minute (Free Tier)
#         # 60 seconds / 15 requests = 4 seconds delay
#         time.sleep(4)

#     # Final save
#     df_processed.to_csv(output_filename, index=False)
#     print(f"\nProcessing Complete! File saved as: {output_filename}")
#     return df_processed

# # --- 4. UTILITY: LOAD PROCESSED DATA ---
# def load_and_fix_lists(filename):
#     """
#     Loads the CSV and converts string-formatted lists back into Python lists.
#     """
#     df = pd.read_csv(filename)
#     # Convert JSON string column back to actual Python Lists
#     df['inclusion_criteria'] = df['inclusion_criteria'].apply(ast.literal_eval)
#     df['exclusion_criteria'] = df['exclusion_criteria'].apply(ast.literal_eval)
#     return df

class ClinicalCriteria(typing.TypedDict):
    inclusion_criteria: list[str]
    exclusion_criteria: list[str]

# --- DEFENSIVE EXTRACTION FUNCTION ---
def extract_criteria_gemini(text: str) -> dict:
    """
    Safely extracts criteria.
    Returns empty lists if API fails or JSON is malformed to prevent crashes.
    """
    # Default structure in case of failure
    default_response = {"inclusion_criteria": [], "exclusion_criteria": []}

    if not text or pd.isna(text):
        return default_response

    prompt = f"Extract clinical trial criteria as JSON from this text: {text}"

    try:
        response = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json",
                response_schema=ClinicalCriteria
            )
        )

        if response and response.text:
            # We use .get() to avoid KeyError even if JSON is valid but keys are missing
            data = json.loads(response.text)
            return {
                "inclusion_criteria": data.get("inclusion_criteria", []),
                "exclusion_criteria": data.get("exclusion_criteria", [])
            }
        return default_response

    except json.JSONDecodeError:
        # If Gemini returns a broken JSON string, we catch it here
        print("JSON Error: Malformed response from API. Skipping row.")
        return default_response
    except Exception as e:
        # Any other API or Network error
        print(f"General API Error: {e}")
        return default_response

# --- ROBUST BATCH PROCESSING ---
def process_clinical_dataset(df: pd.DataFrame, output_filename="processed_studies.csv"):
    """
    Processes the dataframe with error handling and automatic resume (checkpoint).
    """
    DATA_PROCESSED_PATH = '/content/drive/My Drive/Mestrado/Dissertação/mimic-iv-ext-cardiac-disease/processed/'
    output_filename = os.path.join(DATA_PROCESSED_PATH, output_filename)

    # 1. Ensure nct_id is a column, not just an index
    if df.index.name == 'nct_id' or 'nct_id' not in df.columns:
        df = df.reset_index()

    # 1. Resume logic: Check if we already have a partial result
    if os.path.exists(output_filename):
        df_processed = pd.read_csv(output_filename)
        processed_count = len(df_processed)
        print(f"Resuming from index {processed_count}...")
    else:
        df_processed = pd.DataFrame()
        processed_count = 0

    # 2. Loop starting from the last successful index
    for i in tqdm(range(processed_count, len(df))):
        row = df.iloc[i].copy()

        # Safe extraction
        extraction_result = extract_criteria_gemini(row['eligibility_criteria'])

        # We use .get() here as a second layer of safety
        inc = extraction_result.get('inclusion_criteria', [])
        exc = extraction_result.get('exclusion_criteria', [])

        # Save as JSON strings for CSV compatibility
        row['inclusion_criteria'] = json.dumps(inc)
        row['exclusion_criteria'] = json.dumps(exc)

        # Add to the result dataframe
        df_processed = pd.concat([df_processed, pd.DataFrame([row])], ignore_index=True)

        # Save to file every 10 rows (Checkpoint)
        if (i + 1) % 10 == 0:
            df_processed.to_csv(output_filename, index=False)

        # Standard delay for Free Tier (15 RPM)
        time.sleep(4)

    # Final save after loop finishes
    df_processed.to_csv(output_filename, index=False)
    print("\n✅ All rows processed successfully!")
    return df_processed

In [ ]:
# @title Main
API_URL = "https://clinicaltrials.gov/api/v2/studies"

disease_list = [
    'Heart failure', 'Acute myocardial infarction', 'Chronic ischemic heart disease',
    'Atrial fibrillation and flutter', 'Nonrheumatic aortic valve disorders',
    'Other diseases of pericardium', 'Paroxysmal tachycardia', 'Acute pericarditis',
    'Atrioventricular and left bundle-branch block', 'Nonrheumatic mitral valve disorders',
    'Acute and subacute endocarditis', 'Angina pectoris', 'Other cardiac arrhythmias',
    'Cardiomyopathy', 'Acute myocarditis', 'Other acute ischemic heart diseases',
    'Subsequent ST elevation (STEMI) and non-ST elevation (NSTEMI) myocardial infarction',
    'Other conduction disorders', 'Cardiac arrest', 'Nonrheumatic tricuspid valve disorders'
]

clinical_trials_df = get_clinical_studies_data(diseases=disease_list, status='COMPLETED', page_size=1000)

# Extract with regex
# clinical_trials_df[['inclusion_criteria', 'exclusion_criteria']] = clinical_trials_df['eligibility_criteria'].apply(lambda x: pd.Series(process_criteria_robust(x)))
# clinical_trials_df.shape

#  Extract with GenAI
final_df = process_clinical_dataset(clinical_trials_df)

Resuming from index 5...


 43%|████▎     | 426/995 [51:00<1:03:48,  6.73s/it]

JSON Error: Malformed response from API. Skipping row.


 66%|██████▌   | 655/995 [1:22:25<37:29,  6.62s/it]

JSON Error: Malformed response from API. Skipping row.


 84%|████████▍ | 837/995 [1:47:02<19:14,  7.31s/it]

JSON Error: Malformed response from API. Skipping row.


100%|██████████| 995/995 [2:09:52<00:00,  7.83s/it]


✅ All rows processed successfully!


In [ ]:
final_df

,nct_id,official_title,eligibility_criteria,inclusion_criteria,exclusion_criteria
0,NCT03319472,Clinical Identification of Malignant Pleural E...,Inclusion Criteria:\n\n* Pleural effusion\n* H...,"[""Pleural effusion"", ""Hospital admission"", ""No...","[""No diagnosis at one month post-admission"", ""..."
1,NCT00236236,CONTAK RENEWAL® Heart Failure Heart Rate Varia...,Inclusion Criteria:\n\n* Patients receiving th...,"[""Patients receiving their first CRT-D"", ""Pati...","[""Patients who are anticipated to receive paci..."
2,NCT01550107,A Prospective Study to Evaluate the Effect of ...,Inclusion Criteria:\n\nAge 65 and over 6-Minut...,"[""Age 65 and over"", ""6-Minute Walk Distance <4...","[""Documented history of peripheral arterial di..."
3,NCT00356044,Femoral Versus Radial Access for Coronary Inte...,Inclusion Criteria:\n\n* Patients with ST elev...,"[""Patients with ST elevation acute myocardial ...","[""Patients in cardiogenic shock were excluded ..."
4,NCT02805387,"Bicarbonate, a New Treatment of Labour Dystocia","Inclusion Criteria:\n\n* primiparity, singleto...","[""primiparity, singleton pregnancy"", ""with an ...","[""multiparous women"", ""deliveries with non-cep..."
...,...,...,...,...,...
995,NCT02700958,Study of Remote Ischemic Preconditioning as a ...,Inclusion Criteria:\n\n* Age greater than 18 y...,"[""Age greater than 18 years, no upper age limi...","[""Pregnancy"", ""Age less than 18 years"", ""eGFR ..."
996,NCT01110915,Advisa MRI™ System Clinical Investigation,Inclusion Criteria:\n\n* Subjects who have a C...,"[""Subjects who have a Class I or II indication...","[""Subjects with a mechanical tricuspid heart v..."
997,NCT04516525,Assessment of the Impact of Resynchronization ...,Inclusion Criteria:\n\nAdult patients with hea...,"[""Adult patients with heart failure who had me...","[""no written consent to participate in the stu..."
998,NCT01620398,The Brazilian Cardioprotective Nutritional Pro...,Inclusion Criteria:\n\n* Any evidence of coron...,"[""Any evidence of coronary artery disease (CAD...","[""Refusal to provide Informed Consent Statemen..."


In [ ]:
clinical_trials_df

,official_title,eligibility_criteria
nct_id,,
NCT03319472,Clinical Identification of Malignant Pleural E...,Inclusion Criteria:\n\n* Pleural effusion\n* H...
NCT00236236,CONTAK RENEWAL® Heart Failure Heart Rate Varia...,Inclusion Criteria:\n\n* Patients receiving th...
NCT01550107,A Prospective Study to Evaluate the Effect of ...,Inclusion Criteria:\n\nAge 65 and over 6-Minut...
NCT00356044,Femoral Versus Radial Access for Coronary Inte...,Inclusion Criteria:\n\n* Patients with ST elev...
NCT02805387,"Bicarbonate, a New Treatment of Labour Dystocia","Inclusion Criteria:\n\n* primiparity, singleto..."
...,...,...
NCT02700958,Study of Remote Ischemic Preconditioning as a ...,Inclusion Criteria:\n\n* Age greater than 18 y...
NCT01110915,Advisa MRI™ System Clinical Investigation,Inclusion Criteria:\n\n* Subjects who have a C...
NCT04516525,Assessment of the Impact of Resynchronization ...,Inclusion Criteria:\n\nAdult patients with hea...


In [ ]:
# Test the first row
test_text = clinical_trials_df['eligibility_criteria'].iloc[0]
print(extract_criteria_gemini(test_text))

In [ ]:
sample_df

In [ ]:
clinical_trials_df